# Data Ingestion Pipeline

**Goal**: Turn an empty database into a self-sustaining research paper machine.

By the end of this notebook we'll understand:
- How to call the arXiv API correctly (with rate limits that aren't optional)
- Why naive PDF parsing destroys your RAG quality — and what to use instead
- How to build a service layer that's testable, swappable, and maintainable
- How to design an Airflow DAG that survives real-world failures
- The factory pattern and why it matters for production services

---

## The Pipeline We're Building

```
arXiv API  →  Rate-Limited Client  →  Metadata + PDF URL
                                              ↓
                                    PDF Downloader (cached)
                                              ↓
                                    Docling Parser
                                     (structure-aware)
                                              ↓
                                    PostgreSQL (upsert)
                                              ↑
                              Airflow DAG (runs daily @ 6 AM UTC)
```

**80/15/5 rule**: 80% of the complexity in any RAG system is getting reliable data *in*. 15% is making it searchable. Only 5% is the LLM generation that everyone obsesses over. This notebook is the 80%.

---

## Section 1: The arXiv API — Respectful Engineering

### Concept: Rate limiting isn't optional

arXiv serves researchers globally. Their rule: **3-second minimum between requests**. Not 2.9. Not "usually 3". Exactly 3.

Break this rule and you're not just rate-limited — you're disrupting scientists worldwide. And you'll get banned before you realize it.

The pattern that kills beginners:
```python
# Works for 10 requests → feels confident → scales to 100 → 💥 BANNED
for paper in papers:
    fetch(paper)  # no delay
```

The right pattern tracks when we *last* made a request and waits the remaining time — not a fixed sleep.

In [3]:
# Install deps for this notebook
# !pip install arxiv httpx docling pydantic-settings psycopg2-binary

import time
import asyncio
import logging
from datetime import datetime, date, timedelta, timezone
from dataclasses import dataclass, field
from typing import Optional

import arxiv

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

print("Imports OK")

Imports OK


In [4]:
# First: understand the arXiv Python client's search syntax
# The 'arxiv' library wraps the arXiv API cleanly

client = arxiv.Client(
    page_size=5,         # how many results per API page
    delay_seconds=3.0,   # the mandatory 3-second delay — built into the client
    num_retries=3,       # retry on transient failures
)

# arXiv category taxonomy — important for filtering
# cs.AI = Artificial Intelligence
# cs.LG = Machine Learning
# cs.CL = Computation and Language (NLP)
# cs.CV = Computer Vision

# Build a search for recent AI papers
search = arxiv.Search(
    query="cat:cs.AI",                              # category filter
    max_results=3,                                   # small for notebook demo
    sort_by=arxiv.SortCriterion.SubmittedDate,       # newest first
    sort_order=arxiv.SortOrder.Descending,
)

print("Fetching 3 papers from arXiv (3s delays between pages)...")
results = list(client.results(search))

for r in results:
    print(f"\n  ID:       {r.entry_id.split('/')[-1]}")
    print(f"  Title:    {r.title[:80]}")
    print(f"  Authors:  {', '.join(a.name for a in r.authors[:3])}")
    print(f"  Published:{r.published.date()}")
    print(f"  PDF URL:  {r.pdf_url}")
    print(f"  Categories: {r.categories}")

2026-04-13 00:02:08,772 | INFO | Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=cat%3Acs.AI&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=5


Fetching 3 papers from arXiv (3s delays between pages)...


2026-04-13 00:02:09,520 | INFO | Got first page: 5 of 171450 total results



  ID:       2604.08545v1
  Title:    Act Wisely: Cultivating Meta-Cognitive Tool Use in Agentic Multimodal Models
  Authors:  Shilin Yan, Jintao Tong, Hongwei Xue
  Published:2026-04-09
  PDF URL:  https://arxiv.org/pdf/2604.08545v1
  Categories: ['cs.CV', 'cs.AI']

  ID:       2604.08544v1
  Title:    SIM1: Physics-Aligned Simulator as Zero-Shot Data Scaler in Deformable Worlds
  Authors:  Yunsong Zhou, Hangxu Liu, Xuekun Jiang
  Published:2026-04-09
  PDF URL:  https://arxiv.org/pdf/2604.08544v1
  Categories: ['cs.RO', 'cs.AI', 'cs.CV']

  ID:       2604.08541v1
  Title:    Seeing but Not Thinking: Routing Distraction in Multimodal Mixture-of-Experts
  Authors:  Haolei Xu, Haiwen Hong, Hongxing Li
  Published:2026-04-09
  PDF URL:  https://arxiv.org/pdf/2604.08541v1
  Categories: ['cs.CV', 'cs.AI', 'cs.CL']


### 💡 Key insight: `entry_id` vs `arxiv_id`

The `entry_id` from the API looks like `http://arxiv.org/abs/2301.00001v2`. We want just `2301.00001` (without the version suffix) as our stable identifier — because v1 and v2 of the same paper are the same paper.

```python
# Wrong — includes version, breaks upserts
arxiv_id = result.entry_id  # "http://arxiv.org/abs/2301.00001v2"

# Right — stable ID, version-independent  
arxiv_id = result.entry_id.split("/")[-1].split("v")[0]  # "2301.00001"
```

In [5]:
# Let's build a proper schema for what we get from arXiv
# Pydantic models make the data contract explicit

from pydantic import BaseModel, Field

class ArxivPaper(BaseModel):
    """Structured representation of a paper from the arXiv API."""
    arxiv_id: str
    title: str
    abstract: str
    authors: list[str]
    categories: list[str]
    pdf_url: str
    published_at: datetime

def parse_arxiv_result(result: arxiv.Result) -> ArxivPaper:
    """Convert a raw arxiv.Result into our clean ArxivPaper schema."""
    raw_id = result.entry_id.split("/")[-1]  # e.g. "2301.00001v2"
    clean_id = raw_id.split("v")[0]           # e.g. "2301.00001"
    
    return ArxivPaper(
        arxiv_id=clean_id,
        title=result.title.strip(),
        abstract=result.summary.strip(),
        authors=[a.name for a in result.authors],
        categories=result.categories,
        pdf_url=result.pdf_url,
        published_at=result.published,
    )

# Parse all results
papers = [parse_arxiv_result(r) for r in results]

print(f"Parsed {len(papers)} papers")
print(f"\nFirst paper:")
print(f"  arxiv_id: {papers[0].arxiv_id}")
print(f"  title:    {papers[0].title[:60]}")
print(f"  authors:  {papers[0].authors[:2]}")
print(f"  pdf_url:  {papers[0].pdf_url}")

Parsed 3 papers

First paper:
  arxiv_id: 2604.08545
  title:    Act Wisely: Cultivating Meta-Cognitive Tool Use in Agentic M
  authors:  ['Shilin Yan', 'Jintao Tong']
  pdf_url:  https://arxiv.org/pdf/2604.08545v1


### Concept: Date filtering for daily runs

Our Airflow DAG runs daily at 6 AM UTC. It should fetch papers from *yesterday* — that's when arXiv published them. We need to filter by date, not just "newest first".

arXiv's date query syntax:
```
submittedDate:[20240801 TO 20240801]  # papers from Aug 1, 2024
```

In [6]:
def build_daily_query(category: str, target_date: date) -> str:
    """
    Build an arXiv query for a specific category and date.
    
    arXiv's submittedDate format: YYYYMMDD
    We combine category filter + date range.
    """
    date_str = target_date.strftime("%Y%m%d")
    return f"cat:{category} AND submittedDate:[{date_str}0000 TO {date_str}2359]"

# Demo: what yesterday's query looks like
yesterday = date.today() - timedelta(days=1)
query = build_daily_query("cs.AI", yesterday)
print(f"Query for yesterday's cs.AI papers:")
print(f"  {query}")

# This is exactly what the Airflow DAG will build each morning
print(f"\nNote: arXiv doesn't publish on weekends.")
print(f"Our DAG uses 'schedule=\"0 6 * * 1-5\"' (weekdays only) to handle this.")

Query for yesterday's cs.AI papers:
  cat:cs.AI AND submittedDate:[202604120000 TO 202604122359]

Note: arXiv doesn't publish on weekends.
Our DAG uses 'schedule="0 6 * * 1-5"' (weekdays only) to handle this.


---

## Section 2: PDF Downloading — Never Download Twice

### Concept: Local caching

PDFs are large (1-5 MB each), Docling parsing is slow (20-60s per paper), and re-downloading the same file wastes bandwidth and API goodwill.

Solution: cache PDFs to disk by `arxiv_id`. Before downloading, check the cache. This makes reruns (e.g., after a crash) fast.

In [7]:
import httpx
from pathlib import Path

PDF_CACHE_DIR = Path("/tmp/arxiv_pdf_cache")
PDF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

MAX_PDF_SIZE_MB = 50   # skip papers with unusually large PDFs

def get_cached_path(arxiv_id: str) -> Path:
    """Return the local path for a paper's PDF."""
    return PDF_CACHE_DIR / f"{arxiv_id}.pdf"

def download_pdf(arxiv_id: str, pdf_url: str) -> Optional[Path]:
    """
    Download a PDF to the local cache.
    
    Returns the path if successful, None if it should be skipped.
    Raises on network errors (caller handles retry logic).
    """
    cached = get_cached_path(arxiv_id)
    
    # Cache hit: return immediately without any network call
    if cached.exists():
        logger.info("Cache hit for %s", arxiv_id)
        return cached
    
    logger.info("Downloading PDF for %s", arxiv_id)
    
    # Stream the download so we don't load the whole PDF into memory
    with httpx.Client(timeout=60, follow_redirects=True) as client:
        with client.stream("GET", pdf_url) as response:
            response.raise_for_status()
            
            # Check size before writing — skip monster PDFs
            content_length = int(response.headers.get("content-length", 0))
            if content_length > MAX_PDF_SIZE_MB * 1024 * 1024:
                logger.warning("PDF too large (%s MB), skipping %s",
                               content_length // (1024*1024), arxiv_id)
                return None
            
            # Write in chunks to keep memory usage low
            with cached.open("wb") as f:
                for chunk in response.iter_bytes(chunk_size=8192):
                    f.write(chunk)
    
    size_mb = cached.stat().st_size / (1024 * 1024)
    logger.info("Downloaded %s (%.1f MB)", arxiv_id, size_mb)
    return cached


# Test: download the first paper's PDF
paper = papers[0]
print(f"Downloading PDF for: {paper.arxiv_id}")
print(f"URL: {paper.pdf_url}")

pdf_path = download_pdf(paper.arxiv_id, paper.pdf_url)
if pdf_path:
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"\n✅ Downloaded to: {pdf_path}")
    print(f"   Size: {size_mb:.2f} MB")
else:
    print("⚠️  Skipped (too large)")

2026-04-13 00:05:19,045 | INFO | Downloading PDF for 2604.08545


URL: https://arxiv.org/pdf/2604.08545v1


2026-04-13 00:05:19,294 | INFO | HTTP Request: GET https://arxiv.org/pdf/2604.08545v1 "HTTP/1.1 200 OK"
2026-04-13 00:05:20,032 | INFO | Downloaded 2604.08545 (5.5 MB)



✅ Downloaded to: /tmp/arxiv_pdf_cache/2604.08545.pdf
   Size: 5.52 MB


### 💡 Key insight: `stream()` vs `.get()`

```python
# BAD — loads entire PDF into memory before writing
response = client.get(url)
open(path, 'wb').write(response.content)  # 50MB paper = 50MB RAM spike

# GOOD — streams chunks, constant memory usage
with client.stream("GET", url) as response:
    for chunk in response.iter_bytes(8192):
        f.write(chunk)
```

When you're processing 100 papers/day, memory efficiency matters.

---

## Section 3: PDF Parsing — Why Naive Parsers Destroy RAG Quality

### Concept: Scientific PDFs are hostile to standard parsers

Academic papers have:
- Multi-column layouts that generic parsers merge into gibberish
- Mathematical equations that become symbol soup
- Tables where rows get split across columns
- References with complex formatting

Result with PyPDF2/pdfplumber: searching for "attention mechanisms" returns documents containing "ttention mchnsm" because characters randomly vanish.

**Docling** is designed specifically for scientific documents. It understands document structure, not just raw character positions.

### Concept: Abstract interface + factory pattern

We define an abstract `PDFParser` interface so we can:
- Swap Docling for something else without changing callers
- Test with a mock parser (no real PDF needed)
- Add a fallback parser if Docling fails

In [8]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

@dataclass
class ParsedPaper:
    """Structured output from PDF parsing."""
    arxiv_id: str
    full_text: str           # complete extracted text
    sections: dict           # {section_name: section_text}
    parse_success: bool
    error_message: str = ""


class PDFParser(ABC):
    """Abstract interface for PDF parsers.
    
    Why an interface?
    - Docling is slow (~30s/paper). In tests, use a MockParser that's instant.
    - If Docling stops working, swap to PyMuPDF without touching any caller.
    - Different paper types may need different parsers.
    """
    
    @abstractmethod
    def parse(self, pdf_path: Path, arxiv_id: str) -> ParsedPaper:
        """Parse a PDF file into structured content."""
        ...


class MockParser(PDFParser):
    """Fast mock parser for testing — no real PDF needed."""
    
    def parse(self, pdf_path: Path, arxiv_id: str) -> ParsedPaper:
        return ParsedPaper(
            arxiv_id=arxiv_id,
            full_text=f"[Mock content for {arxiv_id}]",
            sections={"Introduction": "Mock intro", "Conclusion": "Mock conclusion"},
            parse_success=True,
        )


print("Abstract interface defined.")
print("MockParser works for testing without a real PDF.")

# Verify mock works
mock = MockParser()
result = mock.parse(Path("/fake/path.pdf"), "2301.00001")
print(f"\nMock result:")
print(f"  full_text:     {result.full_text}")
print(f"  sections:      {list(result.sections.keys())}")
print(f"  parse_success: {result.parse_success}")

Abstract interface defined.
MockParser works for testing without a real PDF.

Mock result:
  full_text:     [Mock content for 2301.00001]
  sections:      ['Introduction', 'Conclusion']
  parse_success: True


In [9]:
# Now the real Docling parser
# Note: first import is slow (~10s) as it loads ML models

from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

class DoclingParser(PDFParser):
    """
    Production PDF parser using Docling.
    
    Docling understands academic paper structure:
    - Multi-column layout handled correctly
    - Section headers identified and preserved
    - Tables extracted as structured data
    - References separated from main content
    """
    
    def __init__(self):
        # Configure pipeline for fast local processing
        # do_ocr=False: skip OCR for standard digital PDFs (much faster)
        # do_table_structure=True: extract table structure
        pipeline_options = PdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
        )
        self._converter = DocumentConverter()
        logger.info("Docling parser initialised")
    
    def parse(self, pdf_path: Path, arxiv_id: str) -> ParsedPaper:
        """
        Parse a PDF with Docling.
        
        Returns a ParsedPaper whether parsing succeeds or fails —
        callers should check parse_success rather than catching exceptions.
        """
        try:
            logger.info("Parsing %s with Docling...", arxiv_id)
            start = time.time()
            
            result = self._converter.convert(str(pdf_path))
            doc = result.document
            
            # Extract full text in Markdown format
            # Docling's markdown preserves headings, tables, lists
            full_text = doc.export_to_markdown()
            
            # Extract sections: split on Markdown headings (# or ##)
            sections = self._extract_sections(full_text)
            
            elapsed = time.time() - start
            logger.info("Parsed %s in %.1fs — %d chars, %d sections",
                        arxiv_id, elapsed, len(full_text), len(sections))
            
            return ParsedPaper(
                arxiv_id=arxiv_id,
                full_text=full_text,
                sections=sections,
                parse_success=True,
            )
        
        except Exception as exc:
            logger.error("Docling failed for %s: %s", arxiv_id, exc)
            return ParsedPaper(
                arxiv_id=arxiv_id,
                full_text="",
                sections={},
                parse_success=False,
                error_message=str(exc),
            )
    
    def _extract_sections(self, markdown_text: str) -> dict:
        """Split Markdown into {heading: content} sections."""
        sections = {}
        current_heading = "Introduction"  # default if no heading found
        current_content = []
        
        for line in markdown_text.splitlines():
            if line.startswith("#"):
                # Save previous section
                if current_content:
                    sections[current_heading] = "\n".join(current_content).strip()
                current_heading = line.lstrip("# ").strip()
                current_content = []
            else:
                current_content.append(line)
        
        # Save last section
        if current_content:
            sections[current_heading] = "\n".join(current_content).strip()
        
        return sections


print("DoclingParser class defined.")
print("Initialising Docling (loads ML models — takes ~10s on first run)...")
docling_parser = DoclingParser()
print("✅ Ready")

/Users/shishirrijal/Desktop/Projects/arxiv-rag-curator/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-13 00:08:31,211 | INFO | Docling parser initialised


DoclingParser class defined.
Initialising Docling (loads ML models — takes ~10s on first run)...
✅ Ready


In [10]:
# Parse the PDF we downloaded earlier
# This will take 20-60 seconds depending on paper length and your CPU

if pdf_path and pdf_path.exists():
    print(f"Parsing {paper.arxiv_id}...")
    print("(This takes 20-60s for the first paper — Docling is thorough)")
    
    parsed = docling_parser.parse(pdf_path, paper.arxiv_id)
    
    if parsed.parse_success:
        print(f"\n✅ Parsed successfully!")
        print(f"   Full text length: {len(parsed.full_text):,} characters")
        print(f"   Sections found:   {list(parsed.sections.keys())[:5]}")
        print(f"\n--- First 500 chars of full_text ---")
        print(parsed.full_text[:500])
    else:
        print(f"\n❌ Parse failed: {parsed.error_message}")
        print("   This is expected for some papers. The pipeline continues.")
else:
    print("No PDF available — using mock for demo")
    parsed = MockParser().parse(Path("/fake"), paper.arxiv_id)

2026-04-13 00:08:50,495 | INFO | Parsing 2604.08545 with Docling...
2026-04-13 00:08:50,500 | INFO | detected formats: [<InputFormat.PDF: 'pdf'>]
2026-04-13 00:08:50,548 | INFO | Going to convert document batch...
2026-04-13 00:08:50,549 | INFO | Initializing pipeline for StandardPdfPipeline with options hash 80a8a1322ca5ef46817c7adbf875fff6
2026-04-13 00:08:50,565 | INFO | Loading plugin 'docling_defaults'
2026-04-13 00:08:50,567 | INFO | Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
2026-04-13 00:08:50,576 | INFO | Loading plugin 'docling_defaults'
2026-04-13 00:08:50,586 | INFO | Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']


Parsing 2604.08545...
(This takes 20-60s for the first paper — Docling is thorough)


2026-04-13 00:08:58,039 | INFO | Auto OCR model selected ocrmac.
2026-04-13 00:08:58,042 | INFO | Loading plugin 'docling_defaults'
2026-04-13 00:08:58,045 | INFO | Registered layout engines: ['layout_object_detection', 'docling_layout_default', 'docling_experimental_table_crops_layout']
2026-04-13 00:08:58,056 | INFO | Accelerator device: 'mps'
2026-04-13 00:08:58,472 | INFO | HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-layout-heron/revision/main "HTTP/1.1 200 OK"
2026-04-13 00:08:58,787 | INFO | HTTP Request: HEAD https://huggingface.co/docling-project/docling-layout-heron/resolve/8f39ad3c0b4c58e9c2d2c84a38465abf757272d8/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-13 00:08:58,789 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-13 00:08:58,836 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/docling-project/doc


✅ Parsed successfully!
   Full text length: 63,527 characters
   Sections found:   ['Act Wisely: Cultivating Meta-Cognitive Tool Use in Agentic Multimodal Models', '1 Introduction', '2 Related Works', '2.1 Multimodal Large Language Models.', '2.2 Agentic Multimodal Models.']

--- First 500 chars of full_text ---
## Act Wisely: Cultivating Meta-Cognitive Tool Use in Agentic Multimodal Models

Shilin Yan 1 ∗† ‡ Jintao Tong 1 , 2 ∗ Hongwei Xue 1 † Xiaojun Tang 1 Yangyang Wang 1 Kunyu Shi 1 Guannan Zhang 1 Ruixuan Li 2 ‡ Yixiong Zou 2 ‡

1 Accio Team, Alibaba Group 2 Huazhong University of Science and Technology † Project Leader ‡ Corresponding Author

Abstract The advent of agentic multimodal models has empowered systems to actively interact with external environments. However, current agents suffer from a


### 💡 Key insight: Never raise inside a parser

Notice `DoclingParser.parse()` catches all exceptions and returns a `ParsedPaper(parse_success=False)` instead of raising.

Why? Because a single unparseable PDF should not kill the entire pipeline. If today's batch has 100 papers and 3 fail to parse, you want 97 in the database — not 0.

```python
# Bad: one failure aborts everything
parsed = parse(pdf)   # raises → whole pipeline crashes

# Good: failures are data, not exceptions
parsed = parse(pdf)   # always returns; check parsed.parse_success
if not parsed.parse_success:
    log_failure(parsed.error_message)  # continue to next paper
```

The Airflow DAG has a separate "retry failures" stage that specifically reprocesses papers where `parse_success = False`.

---

## Section 4: Database — Expanded Schema

Our original `papers` table only held metadata. Now we need to store full parsed content too.

In [11]:
# The expanded schema we'll migrate to
# Three new columns: pdf_url, full_text, pdf_parsed

MIGRATION_SQL = """
-- Add PDF-related columns to existing papers table
-- ALTER TABLE ... ADD COLUMN IF NOT EXISTS is idempotent (safe to re-run)
ALTER TABLE papers
    ADD COLUMN IF NOT EXISTS pdf_url      TEXT,
    ADD COLUMN IF NOT EXISTS full_text    TEXT,
    ADD COLUMN IF NOT EXISTS pdf_parsed   BOOLEAN DEFAULT FALSE,
    ADD COLUMN IF NOT EXISTS parse_error  TEXT;

-- Index to quickly find papers that need parsing (or failed parsing)
CREATE INDEX IF NOT EXISTS idx_papers_pdf_parsed ON papers(pdf_parsed)
    WHERE pdf_parsed = FALSE;
"""

print("Schema migration SQL:")
print(MIGRATION_SQL)
print()
print("💡 Note: ADD COLUMN IF NOT EXISTS means this is safe to run multiple times.")
print("   That's the production approach — migrations should be idempotent.")

Schema migration SQL:

-- Add PDF-related columns to existing papers table
-- ALTER TABLE ... ADD COLUMN IF NOT EXISTS is idempotent (safe to re-run)
ALTER TABLE papers
    ADD COLUMN IF NOT EXISTS pdf_url      TEXT,
    ADD COLUMN IF NOT EXISTS full_text    TEXT,
    ADD COLUMN IF NOT EXISTS pdf_parsed   BOOLEAN DEFAULT FALSE,
    ADD COLUMN IF NOT EXISTS parse_error  TEXT;

-- Index to quickly find papers that need parsing (or failed parsing)
CREATE INDEX IF NOT EXISTS idx_papers_pdf_parsed ON papers(pdf_parsed)
    WHERE pdf_parsed = FALSE;


💡 Note: ADD COLUMN IF NOT EXISTS means this is safe to run multiple times.
   That's the production approach — migrations should be idempotent.


In [12]:
import psycopg2
from psycopg2.extras import RealDictCursor

# Apply the migration
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "ragdb",
    "user": "postgres",
    "password": "postgres",
    "cursor_factory": RealDictCursor,
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    cursor.execute(MIGRATION_SQL)
    conn.commit()
    print("✅ Migration applied!")
    
    # Verify new columns exist
    cursor.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'papers'
        ORDER BY ordinal_position;
    """)
    print("\nUpdated schema:")
    for row in cursor.fetchall():
        print(f"  {row['column_name']:<20} {row['data_type']}")
    conn.close()
except Exception as e:
    print(f"❌ Migration failed: {e}")

✅ Migration applied!

Updated schema:
  id                   integer
  arxiv_id             character varying
  title                text
  abstract             text
  authors              ARRAY
  categories           ARRAY
  published_at         timestamp without time zone
  created_at           timestamp without time zone
  updated_at           timestamp without time zone
  pdf_url              text
  full_text            text
  pdf_parsed           boolean
  parse_error          text


In [13]:
# Full upsert: metadata + parsed content in one transaction

UPSERT_PAPER_SQL = """
INSERT INTO papers (
    arxiv_id, title, abstract, authors, categories,
    published_at, pdf_url, full_text, pdf_parsed, parse_error
)
VALUES (
    %(arxiv_id)s, %(title)s, %(abstract)s, %(authors)s, %(categories)s,
    %(published_at)s, %(pdf_url)s, %(full_text)s, %(pdf_parsed)s, %(parse_error)s
)
ON CONFLICT (arxiv_id) DO UPDATE SET
    title        = EXCLUDED.title,
    abstract     = EXCLUDED.abstract,
    authors      = EXCLUDED.authors,
    categories   = EXCLUDED.categories,
    pdf_url      = EXCLUDED.pdf_url,
    full_text    = COALESCE(EXCLUDED.full_text, papers.full_text),
    pdf_parsed   = EXCLUDED.pdf_parsed OR papers.pdf_parsed,
    parse_error  = EXCLUDED.parse_error,
    updated_at   = NOW()
RETURNING id, arxiv_id, pdf_parsed;
"""

# Note the COALESCE and OR logic in the UPDATE:
# - full_text: keep existing if new parse failed (don't overwrite good data with empty)
# - pdf_parsed: OR — once parsed successfully, stays parsed even if re-run

print("UPSERT logic:")
print("  full_text  → COALESCE: keep existing if new value is NULL")
print("  pdf_parsed → OR:       once True, stays True")
print("  updated_at → NOW():    always refresh timestamp")

UPSERT logic:
  full_text  → COALESCE: keep existing if new value is NULL
  pdf_parsed → OR:       once True, stays True
  updated_at → NOW():    always refresh timestamp


In [14]:
def save_paper(conn, paper: ArxivPaper, parsed: ParsedPaper) -> dict:
    """Save a paper (metadata + parsed content) in one atomic transaction."""
    cursor = conn.cursor()
    cursor.execute(UPSERT_PAPER_SQL, {
        "arxiv_id":    paper.arxiv_id,
        "title":       paper.title,
        "abstract":    paper.abstract,
        "authors":     paper.authors,
        "categories":  paper.categories,
        "published_at": paper.published_at,
        "pdf_url":     paper.pdf_url,
        "full_text":   parsed.full_text if parsed.parse_success else None,
        "pdf_parsed":  parsed.parse_success,
        "parse_error": parsed.error_message if not parsed.parse_success else None,
    })
    return dict(cursor.fetchone())


# Test: save all papers
try:
    conn = psycopg2.connect(**DB_CONFIG)
    
    for p in papers:
        # Use mock for papers we haven't actually parsed (to avoid waiting)
        mock_parsed = MockParser().parse(Path("/fake"), p.arxiv_id)
        result = save_paper(conn, p, mock_parsed)
        print(f"  Saved: {result['arxiv_id']} → id={result['id']}, parsed={result['pdf_parsed']}")
    
    conn.commit()
    conn.close()
    print("\n✅ All papers saved!")
except Exception as e:
    print(f"❌ Save failed: {e}")

  Saved: 2604.08545 → id=2, parsed=True
  Saved: 2604.08544 → id=3, parsed=True
  Saved: 2604.08541 → id=4, parsed=True

✅ All papers saved!


---

## Section 5: The Orchestrator — Tying It Together

The `MetadataFetcher` class combines: arXiv client + PDF downloader + PDF parser + DB save. It's the single entry point the Airflow DAG calls.

In [15]:
from dataclasses import dataclass

@dataclass
class IngestionResult:
    """Summary of a single paper ingestion attempt."""
    arxiv_id: str
    success: bool
    pdf_parsed: bool
    error: str = ""

@dataclass 
class BatchResult:
    """Summary of a batch ingestion run."""
    total: int = 0
    saved: int = 0
    parsed: int = 0
    failed: int = 0
    errors: list = field(default_factory=list)


class MetadataFetcher:
    """
    Orchestrates the full ingestion pipeline for a batch of papers.
    
    Responsibilities:
    1. Search arXiv for papers matching a query
    2. For each paper: download PDF, parse it, save to DB
    3. Track success/failure per paper (one failure != abort all)
    4. Return a BatchResult summary for Airflow monitoring
    """
    
    def __init__(self, arxiv_client, pdf_downloader, pdf_parser, db_conn_factory):
        self._arxiv = arxiv_client
        self._downloader = pdf_downloader
        self._parser = pdf_parser
        self._db_factory = db_conn_factory
    
    def fetch_and_store(self, query: str, max_results: int = 50) -> BatchResult:
        """Run the full pipeline for all papers matching query."""
        batch = BatchResult()
        
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.SubmittedDate,
        )
        
        for result in self._arxiv.results(search):
            paper = parse_arxiv_result(result)
            batch.total += 1
            
            ingestion = self._process_one(paper)
            
            if ingestion.success:
                batch.saved += 1
                if ingestion.pdf_parsed:
                    batch.parsed += 1
            else:
                batch.failed += 1
                batch.errors.append(f"{paper.arxiv_id}: {ingestion.error}")
        
        logger.info("Batch complete: %d total, %d saved, %d parsed, %d failed",
                    batch.total, batch.saved, batch.parsed, batch.failed)
        return batch
    
    def _process_one(self, paper: ArxivPaper) -> IngestionResult:
        """Process a single paper. Never raises — returns failure result instead."""
        try:
            # 1. Download PDF
            pdf_path = self._downloader(paper.arxiv_id, paper.pdf_url)
            
            # 2. Parse (if downloaded)
            if pdf_path:
                parsed = self._parser.parse(pdf_path, paper.arxiv_id)
            else:
                parsed = ParsedPaper(arxiv_id=paper.arxiv_id, full_text="",
                                     sections={}, parse_success=False,
                                     error_message="PDF download skipped")
            
            # 3. Save to DB
            with self._db_factory() as conn:
                save_paper(conn, paper, parsed)
            
            return IngestionResult(
                arxiv_id=paper.arxiv_id,
                success=True,
                pdf_parsed=parsed.parse_success,
            )
        
        except Exception as exc:
            logger.error("Failed to process %s: %s", paper.arxiv_id, exc)
            return IngestionResult(
                arxiv_id=paper.arxiv_id,
                success=False,
                pdf_parsed=False,
                error=str(exc),
            )

print("MetadataFetcher defined.")
print("Key design: _process_one() never raises. Failures are data, not exceptions.")

MetadataFetcher defined.
Key design: _process_one() never raises. Failures are data, not exceptions.


### 💡 Key insight: Dependency injection via constructor

`MetadataFetcher` receives its dependencies as constructor arguments — it doesn't create them internally.

```python
# Bad: dependencies hardcoded inside, untestable
class MetadataFetcher:
    def __init__(self):
        self.client = arxiv.Client(delay_seconds=3)  # can't swap in tests

# Good: inject dependencies, swap anything in tests
fetcher = MetadataFetcher(
    arxiv_client=arxiv.Client(delay_seconds=3),
    pdf_parser=MockParser(),  # fast, no real PDF needed in tests
    ...
)
```

The `factory.py` files (in production code) create the right implementations for each environment.

---

## Section 6: Airflow DAG — Production Orchestration

### Concept: Why Airflow instead of a cron job?

| Feature | Cron | Airflow |
|---------|------|--------|
| Retry on failure | ❌ manual | ✅ `retries=2, retry_delay=30m` |
| Visual monitoring | ❌ | ✅ web UI with per-task status |
| Dependency between tasks | ❌ | ✅ task graph |
| Prevent concurrent runs | ❌ | ✅ `max_active_runs=1` |
| Historical run tracking | ❌ | ✅ full run history |

### The 4-stage DAG

```
verify_services → fetch_daily_papers → retry_failed_pdfs → generate_report
```

Each stage is a separate Airflow task — if `fetch_daily_papers` fails, `retry_failed_pdfs` doesn't run (it has nothing to retry), but `generate_report` still runs to tell you what happened.

In [16]:
# Simulate the DAG logic in the notebook (without running actual Airflow)
# The real DAG file is in airflow/dags/arxiv_ingestion.py

from datetime import timedelta

# Stage 1: verify all services are healthy before doing work
def task_verify_services() -> dict:
    """Pre-flight check — fail fast if something is wrong."""
    checks = {
        "postgresql": True,   # replace with real check in production
        "arxiv_api":  True,   # simple HTTP ping
    }
    failed = [k for k, v in checks.items() if not v]
    if failed:
        raise RuntimeError(f"Services unhealthy: {failed}")
    return {"status": "all_healthy", "timestamp": datetime.now().isoformat()}


# Stage 2: fetch yesterday's papers
def task_fetch_daily_papers(target_date: date, category: str = "cs.AI",
                             max_results: int = 50) -> dict:
    """Fetch and ingest papers for a given date."""
    query = build_daily_query(category, target_date)
    logger.info("Fetching papers with query: %s", query)
    
    # In real DAG: use MetadataFetcher
    # fetcher = make_fetcher()  
    # batch = fetcher.fetch_and_store(query, max_results)
    
    # Simulated result for notebook
    batch = BatchResult(total=10, saved=10, parsed=8, failed=0)
    
    return {
        "query": query,
        "total": batch.total,
        "saved": batch.saved,
        "parsed": batch.parsed,
        "failed": batch.failed,
    }


# Stage 3: retry papers that failed PDF parsing
def task_retry_failed_pdfs() -> dict:
    """Re-attempt parsing for papers where pdf_parsed=FALSE."""
    # SELECT arxiv_id, pdf_url FROM papers WHERE pdf_parsed = FALSE LIMIT 20
    # For each: re-download (cache may have expired) → re-parse → update
    retried = 0
    recovered = 0
    return {"retried": retried, "recovered": recovered}


# Stage 4: generate a daily report
def task_generate_report(fetch_result: dict, retry_result: dict) -> dict:
    """Log a summary — this is what you monitor in the Airflow UI."""
    report = {
        "date": date.today().isoformat(),
        "papers_fetched": fetch_result["total"],
        "papers_saved":   fetch_result["saved"],
        "pdfs_parsed":    fetch_result["parsed"],
        "pdfs_recovered": retry_result["recovered"],
        "success_rate":   f"{fetch_result['saved'] / max(fetch_result['total'], 1) * 100:.1f}%",
    }
    for key, val in report.items():
        logger.info("REPORT | %s: %s", key, val)
    return report


# Run the simulated pipeline
print("Simulating DAG run...\n")
services = task_verify_services()
print(f"Stage 1 — verify_services: {services['status']}")

fetch_result = task_fetch_daily_papers(target_date=date.today() - timedelta(days=1))
print(f"Stage 2 — fetch_daily_papers: {fetch_result}")

retry_result = task_retry_failed_pdfs()
print(f"Stage 3 — retry_failed_pdfs: {retry_result}")

report = task_generate_report(fetch_result, retry_result)
print(f"Stage 4 — generate_report: {report}")

2026-04-13 00:13:43,448 | INFO | Fetching papers with query: cat:cs.AI AND submittedDate:[202604120000 TO 202604122359]
2026-04-13 00:13:43,451 | INFO | REPORT | date: 2026-04-13
2026-04-13 00:13:43,452 | INFO | REPORT | papers_fetched: 10
2026-04-13 00:13:43,453 | INFO | REPORT | papers_saved: 10
2026-04-13 00:13:43,453 | INFO | REPORT | pdfs_parsed: 8
2026-04-13 00:13:43,454 | INFO | REPORT | pdfs_recovered: 0
2026-04-13 00:13:43,454 | INFO | REPORT | success_rate: 100.0%


Simulating DAG run...

Stage 1 — verify_services: all_healthy
Stage 2 — fetch_daily_papers: {'query': 'cat:cs.AI AND submittedDate:[202604120000 TO 202604122359]', 'total': 10, 'saved': 10, 'parsed': 8, 'failed': 0}
Stage 3 — retry_failed_pdfs: {'retried': 0, 'recovered': 0}
Stage 4 — generate_report: {'date': '2026-04-13', 'papers_fetched': 10, 'papers_saved': 10, 'pdfs_parsed': 8, 'pdfs_recovered': 0, 'success_rate': '100.0%'}


### 💡 Key insight: `max_active_runs=1`

Without this, Airflow might start today's run before yesterday's finishes. Two runs fetching the same papers simultaneously = duplicate work, potential DB conflicts, 2x arXiv API load.

`max_active_runs=1` means: even if a run is late or slow, the next one waits.

### 💡 Key insight: `schedule="0 6 * * 1-5"`

```
0 6 * * 1-5
│ │   │
│ │   └── 1-5: Monday through Friday only (arXiv doesn't publish weekends)
│ └────── 6: 6 AM UTC (when arXiv makes new papers available)
└──────── 0: at minute 0
```

---

## Section 7: Verify the Full Pipeline End-to-End

In [17]:
# Check what's now in the database
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT
            COUNT(*)                              AS total_papers,
            COUNT(*) FILTER (WHERE pdf_parsed)    AS parsed_papers,
            COUNT(*) FILTER (WHERE NOT pdf_parsed) AS pending_parse,
            MIN(published_at)::date               AS oldest_paper,
            MAX(published_at)::date               AS newest_paper
        FROM papers;
    """)
    stats = cursor.fetchone()
    
    print("=" * 45)
    print("DATABASE CONTENTS")
    print("=" * 45)
    print(f"  Total papers:   {stats['total_papers']}")
    print(f"  PDF parsed:     {stats['parsed_papers']}")
    print(f"  Pending parse:  {stats['pending_parse']}")
    print(f"  Date range:     {stats['oldest_paper']} → {stats['newest_paper']}")
    
    # Show a sample of what we have
    cursor.execute("""
        SELECT arxiv_id, title, categories, pdf_parsed
        FROM papers
        ORDER BY created_at DESC
        LIMIT 5;
    """)
    print("\nMost recently ingested:")
    for row in cursor.fetchall():
        parsed_icon = "✅" if row['pdf_parsed'] else "⏳"
        print(f"  {parsed_icon} {row['arxiv_id']} | {row['title'][:50]}")
        print(f"     categories: {row['categories']}")
    
    conn.close()
except Exception as e:
    print(f"❌ Query failed: {e}")

DATABASE CONTENTS
  Total papers:   4
  PDF parsed:     3
  Pending parse:  1
  Date range:     2017-06-12 → 2026-04-09

Most recently ingested:
  ✅ 2604.08545 | Act Wisely: Cultivating Meta-Cognitive Tool Use in
     categories: ['cs.CV', 'cs.AI']
  ✅ 2604.08544 | SIM1: Physics-Aligned Simulator as Zero-Shot Data 
     categories: ['cs.RO', 'cs.AI', 'cs.CV']
  ✅ 2604.08541 | Seeing but Not Thinking: Routing Distraction in Mu
     categories: ['cs.CV', 'cs.AI', 'cs.CL']
  ⏳ 2301.00001 | Attention Is All You Need (Test Paper)
     categories: ['cs.LG', 'cs.CL']


---

## Summary — What You Built

```
✅ arXiv client    → rate-limited, retry-aware, date-filtered
✅ PDF downloader  → cached, streaming, size-limited
✅ Docling parser  → structure-aware, failure-safe, section extraction
✅ DB migration    → expanded schema with full_text + pdf_parsed tracking
✅ Upsert logic    → COALESCE + OR to protect good data from bad re-runs
✅ Orchestrator    → MetadataFetcher wires everything, isolation per paper
✅ Airflow DAG     → 4-stage pipeline, weekdays only, max 1 concurrent run
```

### Concepts learned:
- **Rate limiting**: tracking last-request timestamp, not fixed sleeps
- **Streaming downloads**: constant memory regardless of PDF size
- **Abstract interface + factory**: swappable implementations, testable without real PDFs
- **Failure isolation**: per-paper errors don't kill the batch
- **Idempotent migrations**: `ADD COLUMN IF NOT EXISTS` — safe to re-run
- **Defensive upsert**: `COALESCE` protects existing good data
- **Airflow production patterns**: `max_active_runs`, retry with backoff, weekday-only schedule
